# DCF Valuation & Forecast Engine
> **A walkthrough of the financial model — math, code, and outputs in one place.**

This notebook is a self-contained explanation and demonstration of the valuation engine.
It follows the exact same logic as the FastAPI backend, cell by cell, so you can see
every formula, every assumption, and every intermediate result.

**Sections**
1. Setup & Imports
2. Data Sourcing — pulling real financials via `yfinance`
3. Exploratory Data Analysis — understanding the historical financials
4. Revenue Forecasting — projecting top-line growth
5. Cost Projection — the common-size method
6. Income Statement — full reconciled projection
7. Free Cash Flow — building the FCF bridge
8. Discounted Cash Flow — present value of FCFs
9. Terminal Value — Gordon Growth Model
10. EV → Equity Value → Intrinsic Price
11. Sensitivity Analysis — what if our assumptions are wrong?
12. Results Summary & Interpretation

---
> **Note:** Run cells top-to-bottom. Change `TICKER` and the assumption variables
> in Section 1 to model any publicly traded company.


In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# ── Notebook display settings ────────────────────────
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

# ── Plot style ───────────────────────────────────────
plt.rcParams.update({
    'figure.facecolor' : '#0d1117',
    'axes.facecolor'   : '#161b22',
    'axes.edgecolor'   : '#30363d',
    'axes.labelcolor'  : '#7d8590',
    'axes.titlecolor'  : '#e6edf3',
    'xtick.color'      : '#7d8590',
    'ytick.color'      : '#7d8590',
    'text.color'       : '#e6edf3',
    'grid.color'       : '#21262d',
    'grid.linestyle'   : '--',
    'grid.alpha'       : 0.6,
    'font.family'      : 'monospace',
    'figure.dpi'       : 120,
})
ACCENT  = '#58a6ff'
GREEN   = '#3fb950'
RED     = '#f85149'
YELLOW  = '#e3b341'
MUTED   = '#7d8590'

print('All imports OK.')


In [ ]:
# ── MODEL ASSUMPTIONS ────────────────────────────────
# Change these to model any company

TICKER               = 'AAPL'    # Stock ticker (case-insensitive)
REVENUE_GROWTH_RATE  = 0.08      # Annual revenue CAGR (8%)
WACC                 = 0.10      # Weighted Avg Cost of Capital (10%)
TERMINAL_GROWTH_RATE = 0.025     # Perpetuity growth rate (2.5%)
TAX_RATE             = 0.21      # Effective corporate tax rate (21%)
FORECAST_YEARS       = 5         # Explicit forecast horizon
TRAILING_YEARS       = 3         # Years of history used to anchor margins

# Validate the most critical constraint before running anything
assert WACC > TERMINAL_GROWTH_RATE, (
    f'WACC ({WACC:.1%}) must be strictly greater than terminal growth rate '
    f'({TERMINAL_GROWTH_RATE:.1%}). Otherwise Gordon Growth Model denominator <= 0.'
)

print(f'Ticker              : {TICKER}')
print(f'Revenue Growth      : {REVENUE_GROWTH_RATE:.1%} / year')
print(f'WACC                : {WACC:.1%}')
print(f'Terminal Growth Rate: {TERMINAL_GROWTH_RATE:.1%}')
print(f'Tax Rate            : {TAX_RATE:.1%}')
print(f'Forecast Years      : {FORECAST_YEARS}')
print(f'Assumptions valid   : OK')


---
## 2. Data Sourcing

We pull **annual** financial statements via `yfinance`. Three statements are needed:

| Statement | What we get from it |
|---|---|
| Income Statement | Revenue, COGS, Gross Profit, EBIT |
| Cash Flow Statement | D&A (non-cash add-back), CapEx |
| Balance Sheet | Current Assets/Liabilities (Working Capital), Debt, Cash |

**Key normalization steps applied:**
- yfinance returns columns newest-first → we reverse to chronological order
- All values are in raw USD → we divide by 1e9 to work in **billions**
- CapEx is stored as a **negative** number in yfinance (it's a cash outflow). We check
  the median sign and negate only if it's actually negative — preventing double-negation
  for tickers that already report it as positive.
- Some row names differ across tickers (e.g. 'Cost Of Revenue' vs 'Reconciled Cost Of Revenue').
  We try aliases in order and surface a warning if a row is missing entirely.


In [ ]:
t = yf.Ticker(TICKER)

# Fetch raw statements (newest-first columns)
_income_raw   = t.financials      # income statement
_cashflow_raw = t.cashflow        # cash flow statement
_balance_raw  = t.balance_sheet   # balance sheet
_info         = t.info or {}

if _income_raw is None or _income_raw.empty:
    raise ValueError(f'No data for {TICKER}. Check the symbol.')

# Reverse to chronological order (oldest year first)
income   = _income_raw.iloc[:, ::-1].copy()
cashflow = _cashflow_raw.iloc[:, ::-1].copy()
balance  = _balance_raw.iloc[:, ::-1].copy()

# Year labels
years = [str(col.year) for col in income.columns]

print(f'Company      : {_info.get("longName", TICKER)}')
print(f'Years avail. : {years}')
print(f'Current price: ${_info.get("currentPrice", 0):.2f}')


In [ ]:
# ── Safe row getter ──────────────────────────────────
# Returns a Series in billions. Tries alias names in order.
# Returns NaN series (not zero) if no alias is found.

_warnings = []

def get_row(df, *aliases, negate=False):
    """
    Try each alias in order. Return first match, in billions.
    If negate=True AND the median of the series is negative,
    flip sign (handles CapEx sign convention safely).
    """
    for name in aliases:
        if name in df.index:
            s = df.loc[name].astype(float) / 1e9
            if negate and not s.dropna().empty and s.dropna().median() < 0:
                s = -s
            return s
    _warnings.append(f"'{aliases[0]}' not found for {TICKER}. Treating as 0 in margin calculations.")
    return pd.Series(np.nan, index=df.columns)

# ── Extract all required series ──────────────────────
revenue        = get_row(income,   'Total Revenue')
cogs           = get_row(income,   'Cost Of Revenue', 'Cost of Revenue', 'Reconciled Cost Of Revenue')
gross_profit   = get_row(income,   'Gross Profit')
ebit           = get_row(income,   'EBIT', 'Operating Income', 'Ebit')
da             = get_row(cashflow, 'Depreciation And Amortization', 'Depreciation & Amortization', 'Depreciation', 'Reconciled Depreciation')
capex          = get_row(cashflow, 'Capital Expenditure', 'Capital Expenditures', 'Purchase Of PPE', negate=True)
current_assets = get_row(balance,  'Current Assets', 'Total Current Assets')
current_liab   = get_row(balance,  'Current Liabilities', 'Total Current Liabilities')
working_capital = current_assets - current_liab

# Net Debt (most recent year only, for the EV bridge)
bal_latest = balance.iloc[:, -1]
def _scalar(series, *names):
    for n in names:
        if n in series.index:
            v = series[n]
            return float(v) / 1e9 if not np.isnan(v) else 0.0
    return 0.0

total_debt = _scalar(bal_latest, 'Total Debt', 'Long Term Debt And Capital Lease Obligation', 'Long Term Debt')
cash_equiv = _scalar(bal_latest, 'Cash And Cash Equivalents', 'Cash Cash Equivalents And Short Term Investments', 'Cash And Short Term Investments')
net_debt   = total_debt - cash_equiv

# Shares outstanding (diluted, most current)
shares = _info.get('sharesOutstanding') or _info.get('impliedSharesOutstanding')
shares_bn = float(shares) / 1e9 if shares else float(get_row(income, 'Diluted Average Shares', 'Basic Average Shares').iloc[-1])

current_price = float(_info.get('currentPrice') or _info.get('regularMarketPrice') or 0)

if _warnings:
    print('⚠ Data warnings:')
    for w in _warnings: print(f'  {w}')
else:
    print('✓ All financial statement rows found.')

print(f'Net Debt         : ${net_debt:.2f}B')
print(f'Shares (diluted) : {shares_bn:.2f}B')
print(f'Current Price    : ${current_price:.2f}')


---
## 3. Exploratory Data Analysis

Before building any forecast, we examine the historical financials to understand
the business's actual cost structure and cash generation. This is what anchors
the projection assumptions.

**What we look for:**
- Is revenue growth consistent or volatile?
- Are margins (COGS/Rev, EBIT/Rev) stable? Expanding? Contracting?
- Is FCF consistently positive? Is it growing?
- What is the CapEx intensity relative to revenue?
- How does the D&A vs CapEx spread look? (D&A > CapEx suggests asset base is shrinking)


In [ ]:
# ── Historical financials summary table ─────────────
hist = pd.DataFrame({
    'Revenue ($B)'       : revenue,
    'Gross Profit ($B)'  : gross_profit,
    'EBIT ($B)'          : ebit,
    'D&A ($B)'           : da,
    'CapEx ($B)'         : capex,
    'FCF ($B)'           : ebit * (1 - TAX_RATE) + da - capex,  # simplified (no ΔWC for display)
}, index=years)

# Add margin columns
hist['GP Margin']   = (gross_profit / revenue).map('{:.1%}'.format)
hist['EBIT Margin'] = (ebit / revenue).map('{:.1%}'.format)
hist['CapEx/Rev']   = (capex / revenue).map('{:.1%}'.format)

print(f'Historical Financials — {TICKER} (USD Billions)\n')
print(hist.to_string())


In [ ]:
# ── Revenue & EBIT growth chart ──────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f'{TICKER} — Historical Financials', fontsize=13, color='#e6edf3', y=1.02)

# Panel 1: Revenue bar
ax = axes[0]
bars = ax.bar(years, revenue, color=ACCENT, alpha=0.8, width=0.6)
ax.set_title('Revenue ($B)', color='#e6edf3')
ax.set_ylabel('USD Billions', color=MUTED)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fB'))
ax.grid(axis='y')
for bar, val in zip(bars, revenue):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + revenue.max()*0.01,
            f'${val:.1f}B', ha='center', va='bottom', fontsize=8, color='#e6edf3')

# Panel 2: Margins over time
ax = axes[1]
gp_margins   = (gross_profit / revenue * 100).values
ebit_margins = (ebit / revenue * 100).values
x = np.arange(len(years))
ax.plot(years, gp_margins,   marker='o', color=GREEN,  linewidth=2, label='Gross Margin %')
ax.plot(years, ebit_margins, marker='s', color=ACCENT, linewidth=2, label='EBIT Margin %')
ax.set_title('Profitability Margins', color='#e6edf3')
ax.set_ylabel('%', color=MUTED)
ax.legend(fontsize=9)
ax.grid(True)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))

# Panel 3: FCF composition
ax = axes[2]
fcf_approx = (ebit * (1 - TAX_RATE) + da - capex).values
ax.bar(years, da.values,    color=GREEN,  alpha=0.7, label='D&A (add-back)', width=0.5)
ax.bar(years, -capex.values, color=RED,    alpha=0.7, label='CapEx (outflow)', width=0.5, bottom=da.values)
ax.plot(years, fcf_approx, marker='D', color=YELLOW, linewidth=2, label='FCF (approx)', zorder=5)
ax.set_title('D&A vs CapEx vs FCF ($B)', color='#e6edf3')
ax.set_ylabel('USD Billions', color=MUTED)
ax.legend(fontsize=9)
ax.grid(axis='y')
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fB'))

plt.tight_layout()
plt.show()


In [ ]:
# ── Historical CAGR calculation ───────────────────────
rev_arr = revenue.dropna().values
n_years = len(rev_arr) - 1

if n_years > 0 and rev_arr[0] > 0:
    hist_cagr = (rev_arr[-1] / rev_arr[0]) ** (1 / n_years) - 1
    print(f'Historical Revenue CAGR ({years[0]}–{years[-1]}): {hist_cagr:.1%}')
    print(f'Your forecast growth assumption             : {REVENUE_GROWTH_RATE:.1%}')
    diff = REVENUE_GROWTH_RATE - hist_cagr
    direction = 'above' if diff > 0 else 'below'
    print(f'Forecast is {abs(diff):.1%} {direction} historical CAGR')
else:
    print('Cannot compute CAGR — insufficient data.')


---
## 4. Revenue Forecasting

**Method: Constant CAGR (user-supplied)**

```
Revenue(t) = Revenue(0) × (1 + g)^t
```

Where `Revenue(0)` is the most recent annual revenue, `g` is the user-supplied
growth rate, and `t` is the year index (1, 2, 3, ...).

**Why a flat CAGR?**
A flat CAGR is transparent and auditable. More sophisticated approaches
(segment modeling, analyst consensus, ARIMA) add parameters without improving
explainability. You can defend a CAGR clearly in a conversation; you cannot
easily explain why an ARIMA(2,1,1) produced a specific number.

**Setting this assumption:** Use the historical CAGR computed above as an anchor.
Adjust up or down based on your view of the business's trajectory.
Then use the sensitivity table (Section 11) to understand how much this choice matters.


In [ ]:
base_revenue = float(revenue.iloc[-1])
base_year    = int(years[-1])

forecast_years_list = list(range(base_year + 1, base_year + FORECAST_YEARS + 1))
t_values            = list(range(1, FORECAST_YEARS + 1))

projected_revenue = [base_revenue * (1 + REVENUE_GROWTH_RATE) ** t for t in t_values]

rev_df = pd.DataFrame({
    'Year'          : [str(y) for y in forecast_years_list],
    'Revenue ($B)'  : projected_revenue,
    'YoY Growth'    : [REVENUE_GROWTH_RATE] * FORECAST_YEARS,
})
rev_df['YoY Growth'] = rev_df['YoY Growth'].map('{:.1%}'.format)
rev_df = rev_df.set_index('Year')

print(f'Base Revenue ({years[-1]}): ${base_revenue:.2f}B')
print(f'Growth Rate            : {REVENUE_GROWTH_RATE:.1%} per year')
print()
print(rev_df.to_string())


In [ ]:
# ── Revenue chart: historical + projected ────────────
fig, ax = plt.subplots(figsize=(12, 4.5))

# Historical bars
ax.bar(years, revenue.values, color=ACCENT, alpha=0.7, width=0.6, label='Historical')

# Projected bars
proj_years_str = [str(y) for y in forecast_years_list]
ax.bar(proj_years_str, projected_revenue, color=GREEN, alpha=0.7, width=0.6, label='Projected')

# Divider
ax.axvline(x=len(years) - 0.5, color=MUTED, linestyle='--', linewidth=1, alpha=0.6)
ax.text(len(years) - 0.4, max(max(revenue.values), max(projected_revenue)) * 0.95,
        '← Actual  |  Forecast →', color=MUTED, fontsize=9)

ax.set_title(f'{TICKER} Revenue — Historical vs. Projected (USD Billions)', color='#e6edf3', fontsize=12)
ax.set_ylabel('USD Billions', color=MUTED)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fB'))
ax.legend()
ax.grid(axis='y')
plt.tight_layout()
plt.show()


---
## 5. Cost Projection — The Common-Size Method

**Core idea:** Express each cost line as a percentage of revenue (a 'margin'),
compute the trailing average over the last `TRAILING_YEARS` years,
and multiply by projected revenue.

```
Margin(X) = average( X(t) / Revenue(t) )  for t in [last N years]
Projected X(t) = Projected Revenue(t) × Margin(X)
```

This is called **common-size analysis** and is standard in equity research.
Costs scale with revenue automatically, which is the correct assumption
for most operating expense lines.

**Why trailing average and not just the most recent year?**
A single year can be distorted by one-off events (write-downs, restructuring charges,
pandemic effects). A 3-year average gives a smoothed view of the normalized cost structure.

**Critical detail — OpEx is *derived*, not independently projected:**

```
OpEx = Gross Profit − EBIT   (always true by definition)
```

If you independently project COGS and EBIT margins, the implied OpEx won't
reconcile with `Gross Profit − EBIT` in the projections.
Deriving OpEx from history as `GP − EBIT` and then projecting *that* margin
ensures the income statement is always an exact identity chain.


In [ ]:
# ── Compute trailing average margins ─────────────────
def safe_margin(num_series, denom_series, n=TRAILING_YEARS):
    """
    Average ratio over the last n years, ignoring NaN.
    Using the last n years (not all history) to stay anchored
    to the recent business state rather than older periods.
    """
    d = denom_series.iloc[-n:]
    nu = num_series.iloc[-n:]
    ratios = (nu / d).replace([np.inf, -np.inf], np.nan).dropna()
    return float(ratios.mean()) if not ratios.empty else 0.0

# Derive OpEx from history: GP - EBIT (never project independently)
opex_hist = gross_profit - ebit

cogs_margin  = safe_margin(cogs,          revenue)
opex_margin  = safe_margin(opex_hist,     revenue)
ebit_margin  = safe_margin(ebit,          revenue)   # derived reference only
da_margin    = safe_margin(da,            revenue)
capex_margin = safe_margin(capex,         revenue)
wc_margin    = safe_margin(working_capital, revenue)

print(f'Trailing {TRAILING_YEARS}-year average margins used in projection:')
print(f'  COGS  / Revenue : {cogs_margin:.1%}')
print(f'  OpEx  / Revenue : {opex_margin:.1%}   (derived: GP - EBIT)')
print(f'  EBIT  / Revenue : {ebit_margin:.1%}   (reference — not independently projected)')
print(f'  D&A   / Revenue : {da_margin:.1%}')
print(f'  CapEx / Revenue : {capex_margin:.1%}')
print(f'  WC    / Revenue : {wc_margin:.1%}')
print()
print(f'Reconciliation check: COGS+OpEx+EBIT ≈ 100%?',
      f'{cogs_margin + opex_margin + ebit_margin:.1%}',
      '← should equal 100%')


In [ ]:
# ── Visualise historical margins ─────────────────────
fig, ax = plt.subplots(figsize=(11, 4))

gp_m   = (gross_profit / revenue * 100).values
ebit_m = (ebit         / revenue * 100).values
da_m   = (da           / revenue * 100).values
cx_m   = (capex        / revenue * 100).values

ax.plot(years, gp_m,   marker='o', color=GREEN,  linewidth=2, label=f'GP Margin  (avg {1-cogs_margin:.1%})')
ax.plot(years, ebit_m, marker='s', color=ACCENT, linewidth=2, label=f'EBIT Margin (avg {ebit_margin:.1%})')
ax.plot(years, da_m,   marker='^', color=YELLOW, linewidth=1.5, linestyle='--', label=f'D&A/Rev (avg {da_margin:.1%})')
ax.plot(years, cx_m,   marker='v', color=RED,    linewidth=1.5, linestyle='--', label=f'CapEx/Rev (avg {capex_margin:.1%})')

# Horizontal reference lines at trailing averages
ax.axhline((1-cogs_margin)*100, color=GREEN,  linestyle=':', alpha=0.4)
ax.axhline(ebit_margin*100,     color=ACCENT, linestyle=':', alpha=0.4)

ax.set_title(f'{TICKER} — Historical Margin Trends (%) — dashed lines = trailing {TRAILING_YEARS}yr avg',
             color='#e6edf3', fontsize=11)
ax.set_ylabel('%', color=MUTED)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.1f%%'))
ax.legend(fontsize=9)
ax.grid(True)
plt.tight_layout()
plt.show()


---
## 6. Projected Income Statement

The income statement must satisfy these identities **exactly** in every projected year:

```
Revenue
  − COGS              = Revenue × cogs_margin
  ─────────────────────────────────────────────
  = Gross Profit       ← identity: Revenue − COGS
  − OpEx               = Revenue × opex_margin
  ─────────────────────────────────────────────
  = EBIT               ← identity: Gross Profit − OpEx
  × (1 − tax_rate)
  ─────────────────────────────────────────────
  = NOPAT
```

Note: `EBIT` is **not** computed as `Revenue × ebit_margin` — it is the *result*
of subtracting OpEx from Gross Profit. The `ebit_margin` figure is kept for
reference only. This is the fix that makes the income statement a true identity chain.


In [ ]:
# ── Build projected income statement ─────────────────
income_stmt = []

for t, (yr, rev) in enumerate(zip(forecast_years_list, projected_revenue), start=1):
    cogs_proj        = rev * cogs_margin
    gp_proj          = rev - cogs_proj           # identity
    opex_proj        = rev * opex_margin
    ebit_proj        = gp_proj - opex_proj       # identity (NOT rev * ebit_margin)
    nopat_proj       = ebit_proj * (1 - TAX_RATE)

    # Verify reconciliation (should always be True)
    check = abs(ebit_proj - (gp_proj - opex_proj)) < 1e-9

    income_stmt.append({
        'Year'         : str(yr),
        'Revenue'      : rev,
        'COGS'         : cogs_proj,
        'Gross Profit' : gp_proj,
        'GP Margin'    : f'{gp_proj/rev:.1%}',
        'OpEx'         : opex_proj,
        'EBIT'         : ebit_proj,
        'EBIT Margin'  : f'{ebit_proj/rev:.1%}',
        'NOPAT'        : nopat_proj,
        'Reconciles'   : '✓' if check else '✗ ERROR',
    })

is_df = pd.DataFrame(income_stmt).set_index('Year')

# Display numeric columns only (formatted)
numeric_cols = ['Revenue','COGS','Gross Profit','OpEx','EBIT','NOPAT']
display_df = is_df[numeric_cols + ['GP Margin','EBIT Margin','Reconciles']]

print(f'Projected Income Statement — {TICKER} (USD Billions)\n')
print(display_df.to_string())


---
## 7. Free Cash Flow

**Formula:**
```
FCF = NOPAT  +  D&A  −  CapEx  −  ΔWorking Capital
```

| Term | Sign | Why |
|---|---|---|
| NOPAT | + | Unlevered cash earnings (no interest expense) |
| D&A | + | Non-cash charge — reduced EBIT but no cash left the business |
| CapEx | − | Real cash spent on assets, not reflected in EBIT |
| ΔWorking Capital | − (if WC grows) | Growing WC ties up cash — it's a use of funds |

**Why NOPAT instead of Net Income?**
DCF values the business *independent of its capital structure*.
Interest expense pays debt holders — it should not reduce the cash flows
available to *all* investors. NOPAT deliberately excludes interest.
The effect of debt enters later, in the EV → Equity bridge via Net Debt.

**Working Capital sign convention:**
```
ΔWC(t) = WC(t) − WC(t−1)
ΔWC > 0  →  WC grew  →  cash OUTFLOW  →  subtract from FCF
ΔWC < 0  →  WC shrank →  cash INFLOW   →  adds to FCF
```
For the first forecast year, `ΔWC` is computed vs. the last historical WC value.


In [ ]:
# ── Build FCF bridge ─────────────────────────────────
base_wc   = float(working_capital.iloc[-1]) if not working_capital.isna().all() else 0.0
prev_wc   = base_wc
pv_cumsum = 0.0

fcf_bridge = []

for t, (yr, rev) in enumerate(zip(forecast_years_list, projected_revenue), start=1):
    # Income statement (same as Section 6 — recomputing inline for clarity)
    cogs_p  = rev * cogs_margin
    gp_p    = rev - cogs_p
    opex_p  = rev * opex_margin
    ebit_p  = gp_p - opex_p
    nopat_p = ebit_p * (1 - TAX_RATE)

    # Cash flow items
    da_p    = rev * da_margin
    capex_p = rev * capex_margin

    # Working Capital
    wc_proj  = rev * wc_margin
    delta_wc = wc_proj - prev_wc     # positive = cash outflow
    prev_wc  = wc_proj

    # FCF
    fcf = nopat_p + da_p - capex_p - delta_wc

    # Discount to present value
    discount_factor = (1 + WACC) ** t
    pv_fcf          = fcf / discount_factor
    pv_cumsum      += pv_fcf

    fcf_bridge.append({
        'Year'           : str(yr),
        'NOPAT'          : nopat_p,
        '+ D&A'          : da_p,
        '− CapEx'        : capex_p,
        '− ΔWC'          : delta_wc,
        '= FCF'          : fcf,
        'Disc. Factor'   : f'{discount_factor:.3f}×',
        'PV(FCF)'        : pv_fcf,
        'Cumulative PV'  : pv_cumsum,
    })

fcf_df = pd.DataFrame(fcf_bridge).set_index('Year')
print(f'FCF Bridge — {TICKER} (USD Billions)\n')
print(fcf_df.to_string())


In [ ]:
# ── FCF chart: projected + discounted ────────────────
fcf_vals  = [r['= FCF']    for r in fcf_bridge]
pv_vals   = [r['PV(FCF)']  for r in fcf_bridge]
yr_labels = [r['Year']     for r in fcf_bridge]

# Include historical FCF (approximate) for context
hist_fcf = (ebit * (1 - TAX_RATE) + da - capex).values.tolist()

all_labels = years + yr_labels
all_fcf    = hist_fcf + [None] * len(fcf_vals)
proj_fcf   = [None] * len(years) + fcf_vals
disc_fcf   = [None] * len(years) + pv_vals

fig, ax = plt.subplots(figsize=(13, 5))

x = np.arange(len(all_labels))
w = 0.38

# Historical bars
hist_vals_plot = [v if v is not None else 0 for v in all_fcf]
proj_vals_plot = [v if v is not None else 0 for v in proj_fcf]

ax.bar(x - w/2, [v or 0 for v in all_fcf],  width=w, color=ACCENT, alpha=0.75, label='Historical FCF')
ax.bar(x + w/2, [v or 0 for v in proj_fcf], width=w, color=GREEN,  alpha=0.75, label='Projected FCF')

# Discounted FCF line
disc_x = [i for i, v in enumerate(disc_fcf) if v is not None]
disc_y = [v for v in disc_fcf if v is not None]
ax.plot([x[i] for i in disc_x], disc_y, color=YELLOW,
        marker='D', linewidth=2, markersize=5, label='Discounted FCF (PV)', zorder=5)

# Divider between historical and projected
divider_x = len(years) - 0.5
ax.axvline(divider_x, color=MUTED, linestyle='--', linewidth=1, alpha=0.6)
ax.text(divider_x + 0.1, max(filter(None, fcf_vals + hist_fcf)) * 0.9,
        'PROJECTED →', color=MUTED, fontsize=9)

ax.set_xticks(x)
ax.set_xticklabels(all_labels, rotation=0)
ax.set_title(f'{TICKER} — Free Cash Flow: Historical vs. Projected (USD Billions)',
             color='#e6edf3', fontsize=12)
ax.set_ylabel('USD Billions', color=MUTED)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fB'))
ax.legend()
ax.grid(axis='y')
plt.tight_layout()
plt.show()


---
## 8. Discounted Cash Flow (DCF)

The sum of all discounted FCFs over the explicit forecast period:

```
PV(FCFs) = Σ  FCF(t) / (1 + WACC)^t     for t = 1 to n
```

The discount factor `(1 + WACC)^t` represents the opportunity cost of waiting.
A dollar of FCF received 5 years from now is worth less than a dollar today
because of (1) time preference and (2) the riskiness of receiving it.

**WACC — what it represents:**
WACC is the blended required return across all capital providers:
```
WACC = (E/V) × Re  +  (D/V) × Rd × (1 − tax_rate)
```
Where `Re` = cost of equity (e.g. CAPM: Rf + β × ERP), `Rd` = cost of debt,
`E/V` and `D/V` = equity and debt weights.
This model accepts WACC as a user input to keep assumptions transparent and auditable.


In [ ]:
# Sum of PV(FCFs) — already computed incrementally in Section 7
sum_pv_fcf = fcf_df['Cumulative PV'].iloc[-1]

print(f'Sum of PV(FCFs) over {FORECAST_YEARS}-year forecast: ${sum_pv_fcf:.2f}B')
print()
print('Discount factors applied:')
for t in range(1, FORECAST_YEARS + 1):
    df_val = (1 + WACC) ** t
    print(f'  Year {t}: 1 / (1 + {WACC:.1%})^{t} = 1 / {df_val:.3f} = {1/df_val:.4f}')


---
## 9. Terminal Value — Gordon Growth Model

The Terminal Value (TV) captures all cash flows *beyond* the explicit forecast period.
It uses the **Gordon Growth Model** (perpetuity with constant growth):

```
TV = FCF(n) × (1 + g) / (WACC − g)

PV(TV) = TV / (1 + WACC)^n
```

Where:
- `FCF(n)` = Free Cash Flow in the last forecast year
- `g` = terminal growth rate — the assumed rate of FCF growth *forever* after year n
- `WACC` = discount rate
- `n` = number of forecast years

**Critical constraint: WACC > g (strictly)**
If `g ≥ WACC`, the denominator `(WACC − g)` becomes zero or negative,
and the formula produces an infinite or negative terminal value — mathematically
nonsensical. This is validated before running.

**What is a reasonable terminal growth rate?**
The upper bound is the long-run nominal GDP growth rate of the economy
(~2.5–3.5% for developed markets). No single company can permanently grow
faster than the entire economy — it would eventually become larger than it.
Most models use 2–3%.

**Why TV dominates EV:**
Even on a 5-year forecast, Terminal Value typically represents 70–85% of Enterprise Value.
This is not a flaw — it reflects the mathematical reality that most value is
in the long run. It *is* a reason to treat TV assumptions carefully and to
always show the sensitivity analysis.


In [ ]:
fcf_final = fcf_df['= FCF'].iloc[-1]
n         = FORECAST_YEARS

# Gordon Growth Model
denominator    = WACC - TERMINAL_GROWTH_RATE
terminal_value = fcf_final * (1 + TERMINAL_GROWTH_RATE) / denominator
pv_tv          = terminal_value / (1 + WACC) ** n

enterprise_value = sum_pv_fcf + pv_tv
tv_pct_of_ev     = pv_tv / enterprise_value

print('─' * 52)
print(f'Terminal Value Calculation')
print('─' * 52)
print(f'  FCF in final forecast year (FCF_n) : ${fcf_final:.2f}B')
print(f'  Terminal growth rate (g)            : {TERMINAL_GROWTH_RATE:.2%}')
print(f'  WACC                                : {WACC:.2%}')
print(f'  Denominator (WACC − g)              : {denominator:.2%}')
print(f'  Terminal Value (undiscounted)        : ${terminal_value:.2f}B')
print(f'  Discount factor (1+WACC)^{n}         : {(1+WACC)**n:.4f}')
print(f'  PV of Terminal Value                 : ${pv_tv:.2f}B')
print('─' * 52)
print(f'  Sum PV(FCFs)                         : ${sum_pv_fcf:.2f}B')
print(f'  + PV(Terminal Value)                 : ${pv_tv:.2f}B')
print(f'  = Enterprise Value                   : ${enterprise_value:.2f}B')
print('─' * 52)
print(f'  Terminal Value as % of EV            : {tv_pct_of_ev:.1%}')

if tv_pct_of_ev > 0.70:
    print(f'  ⚠  TV > 70% of EV — high sensitivity to terminal assumptions.')
    print(f'     Small changes in WACC or g will materially shift intrinsic price.')
    print(f'     Review the sensitivity analysis in Section 11.')


In [ ]:
# ── EV composition pie chart ─────────────────────────
fig, ax = plt.subplots(figsize=(6, 6))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')

sizes  = [sum_pv_fcf, pv_tv]
labels = [f'PV of FCFs\n${sum_pv_fcf:.1f}B\n({sum_pv_fcf/enterprise_value:.1%})',
          f'PV of Terminal Value\n${pv_tv:.1f}B\n({tv_pct_of_ev:.1%})']
colors = [ACCENT, GREEN]
explode = [0.03, 0.03]

wedges, texts = ax.pie(sizes, labels=labels, colors=colors, explode=explode,
                        startangle=90, textprops={'color': '#e6edf3', 'fontsize': 10},
                        wedgeprops={'linewidth': 1.5, 'edgecolor': '#0d1117'})

ax.set_title(f'{TICKER} Enterprise Value Composition\n${enterprise_value:.1f}B total',
             color='#e6edf3', fontsize=12, pad=15)
plt.tight_layout()
plt.show()


---
## 10. EV → Equity Value → Intrinsic Price

Enterprise Value represents the value of the *entire business* — available to
all capital providers (both debt and equity). To find what belongs to *shareholders*,
we subtract the claims of debt holders:

```
Equity Value  =  Enterprise Value  −  Net Debt
```

Then, to get the per-share intrinsic price:
```
Intrinsic Price  =  Equity Value  /  Diluted Shares Outstanding
```

**Net Debt = Total Debt − Cash**
- If Net Debt > 0 (more debt than cash): equity value is *less* than EV
- If Net Debt < 0 (net cash position): equity value *exceeds* EV — cash accrues to shareholders

**Why diluted shares?**
Diluted shares include stock options, warrants, and convertible instruments.
These represent real obligations to issue equity. Using basic shares only
would overstate intrinsic price per share.


In [ ]:
equity_value    = enterprise_value - net_debt
intrinsic_price = equity_value / shares_bn
upside          = (intrinsic_price - current_price) / current_price if current_price > 0 else 0

print('=' * 54)
print(f' EV → EQUITY VALUE → INTRINSIC PRICE BRIDGE')
print('=' * 54)
print(f'  Σ PV of Forecast FCFs         : ${sum_pv_fcf:.2f}B')
print(f'  + PV of Terminal Value        : ${pv_tv:.2f}B')
print(f'  ─────────────────────────────────────────')
print(f'  = Enterprise Value            : ${enterprise_value:.2f}B')
print(f'  − Net Debt                    : ${net_debt:.2f}B')
print(f'  ─────────────────────────────────────────')
print(f'  = Equity Value                : ${equity_value:.2f}B')
print(f'  ÷ Diluted Shares Outstanding  : {shares_bn:.2f}B shares')
print(f'  ─────────────────────────────────────────')
print(f'  = Intrinsic Price             : ${intrinsic_price:.2f}')
print('=' * 54)
print(f'  Current Market Price          : ${current_price:.2f}')
sign_str = '+' if upside >= 0 else ''
verdict  = 'UNDERVALUED' if upside > 0.05 else ('OVERVALUED' if upside < -0.05 else 'FAIRLY VALUED')
print(f'  Upside / Downside             : {sign_str}{upside:.1%}  →  {verdict}')
print('=' * 54)


In [ ]:
# ── Waterfall / bridge chart ─────────────────────────
labels = ['PV(FCFs)', '+ PV(TV)', '= EV', '− Net Debt', '= Equity Value']
values = [sum_pv_fcf, pv_tv, 0, -net_debt, 0]   # 0 = totals (running sum display)
running = [0, sum_pv_fcf, enterprise_value, enterprise_value, equity_value]
bar_vals = [sum_pv_fcf, pv_tv, enterprise_value, -net_debt, equity_value]
bar_bots = [0,          sum_pv_fcf, 0,            enterprise_value, 0]
bar_cols = [ACCENT, GREEN, '#79c0ff', RED if net_debt > 0 else GREEN, '#56d364']

fig, ax = plt.subplots(figsize=(10, 5))

for i, (lbl, val, bot, col) in enumerate(zip(labels, bar_vals, bar_bots, bar_cols)):
    ax.bar(i, val, bottom=bot, color=col, alpha=0.85, width=0.5, edgecolor='#0d1117', linewidth=1.2)
    total_top = bot + val
    ax.text(i, total_top + enterprise_value * 0.01, f'${total_top:.1f}B',
            ha='center', va='bottom', fontsize=9, color='#e6edf3', fontweight='bold')

# Connector lines
for i in range(len(labels) - 1):
    y_right = bar_bots[i] + bar_vals[i]
    ax.plot([i + 0.25, i + 0.75], [y_right, y_right],
            color=MUTED, linewidth=0.8, linestyle='--', alpha=0.5)

ax.set_xticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=10)
ax.set_title(f'{TICKER} — Enterprise Value Bridge (USD Billions)', color='#e6edf3', fontsize=12)
ax.set_ylabel('USD Billions', color=MUTED)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('$%.0fB'))
ax.grid(axis='y')
plt.tight_layout()
plt.show()


---
## 11. Sensitivity Analysis

A single DCF output is a *point estimate* under one specific set of assumptions.
It should never be treated as a precise number. The sensitivity analysis makes
the *range* of outcomes explicit.

**We vary the two assumptions that drive Terminal Value most:**
- **WACC** — rows (7% to 13%)
- **Terminal Growth Rate** — columns (1% to 4%)

For each `(WACC, TGR)` combination:
1. Re-discount the same FCF stream with the new WACC
2. Recompute Terminal Value with new WACC and TGR
3. Re-run the EV → equity → price bridge

**Heatmap coloring is anchored to the current market price** — not to the
matrix's own min/max. This means:
- 🟢 Green = intrinsic price implies the stock is *undervalued* at those assumptions
- 🔴 Red = intrinsic price implies the stock is *overvalued*
- ⬜ Neutral = approximately fairly valued (within ±10%)

Anchoring to market price gives honest, absolute signal.
Anchoring to matrix min/max would only show *relative* variation.


In [ ]:
wacc_range = [0.07, 0.08, 0.09, 0.10, 0.11, 0.12, 0.13]
tgr_range  = [0.01, 0.015, 0.02, 0.025, 0.03, 0.035, 0.04]

def dcf_price(wacc_val, tgr_val):
    """
    Compute intrinsic price for a given (WACC, TGR) pair.
    Re-discounts the same FCF stream — FCFs themselves don't change
    (they depend on the income statement, not the discount rate).
    Only the discounting and terminal value change.
    """
    if wacc_val <= tgr_val:
        return np.nan   # invalid: Gordon Growth Model requires WACC > TGR

    # Re-discount FCFs with new WACC
    fcf_list = fcf_df['= FCF'].values
    pv_fcfs  = sum(fcf / (1 + wacc_val)**(t+1) for t, fcf in enumerate(fcf_list))

    # Recompute Terminal Value
    fcf_n  = fcf_list[-1]
    n_yrs  = len(fcf_list)
    tv     = fcf_n * (1 + tgr_val) / (wacc_val - tgr_val)
    pv_tv_ = tv / (1 + wacc_val) ** n_yrs

    ev_   = pv_fcfs + pv_tv_
    eq_   = ev_ - net_debt
    price = eq_ / shares_bn if shares_bn > 0 else 0.0
    return round(price, 2)

# Build 2D matrix
matrix = [[dcf_price(w, g) for g in tgr_range] for w in wacc_range]
sens_df = pd.DataFrame(matrix,
    index   = [f'{w:.0%}' for w in wacc_range],
    columns = [f'{g:.1%}' for g in tgr_range]
)
sens_df.index.name   = 'WACC \\ TGR'

print('Sensitivity Matrix — Intrinsic Price per Share ($)\n')
print(sens_df.to_string())
print(f'\nBase case (WACC={WACC:.0%}, TGR={TERMINAL_GROWTH_RATE:.1%}): ${dcf_price(WACC, TERMINAL_GROWTH_RATE):.2f}')
print(f'Current market price: ${current_price:.2f}')


In [ ]:
# ── Sensitivity heatmap ──────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))

# Build color matrix anchored to current market price
# Green = undervalued (intrinsic > market), Red = overvalued
price_matrix = np.array(matrix, dtype=float)

# Custom colormap: red → neutral → green
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
cmap = LinearSegmentedColormap.from_list(
    'dcf_heat',
    ['#f85149', '#4d1a18', '#21262d', '#1a4726', '#3fb950'],
    N=256
)

# Normalize so that current_price = midpoint (white/neutral)
min_p = np.nanmin(price_matrix)
max_p = np.nanmax(price_matrix)
norm  = TwoSlopeNorm(vmin=min_p, vcenter=current_price if current_price > 0 else (min_p+max_p)/2, vmax=max_p)

im = ax.imshow(price_matrix, cmap=cmap, norm=norm, aspect='auto')

# Annotate cells
for i in range(len(wacc_range)):
    for j in range(len(tgr_range)):
        val = price_matrix[i, j]
        if np.isnan(val):
            txt = 'N/A'
            col = MUTED
        else:
            txt = f'${val:.0f}'
            col = '#e6edf3'
        # Highlight base case
        is_base = (abs(wacc_range[i] - WACC) < 0.001 and
                   abs(tgr_range[j] - TERMINAL_GROWTH_RATE) < 0.001)
        weight = 'bold' if is_base else 'normal'
        fontsize = 10 if is_base else 9
        ax.text(j, i, txt, ha='center', va='center',
                color=col, fontsize=fontsize, fontweight=weight)
        if is_base:
            ax.add_patch(mpatches.FancyBboxPatch(
                (j-0.48, i-0.48), 0.96, 0.96,
                boxstyle='round,pad=0.05',
                linewidth=2, edgecolor=ACCENT, facecolor='none'
            ))

ax.set_xticks(range(len(tgr_range)))
ax.set_xticklabels([f'{g:.1%}' for g in tgr_range])
ax.set_yticks(range(len(wacc_range)))
ax.set_yticklabels([f'{w:.0%}' for w in wacc_range])
ax.set_xlabel('Terminal Growth Rate (g)', color=MUTED, fontsize=11)
ax.set_ylabel('WACC', color=MUTED, fontsize=11)
ax.set_title(
    f'{TICKER} — Sensitivity Analysis: Intrinsic Price vs. Market Price ${current_price:.2f}\n'
    f'[■ = base case  |  green = undervalued  |  red = overvalued  |  anchored to market price]',
    color='#e6edf3', fontsize=11, pad=12
)

cbar = plt.colorbar(im, ax=ax, fraction=0.03, pad=0.02)
cbar.set_label('Intrinsic Price ($)', color=MUTED)
cbar.ax.yaxis.set_tick_params(color=MUTED)
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color=MUTED)

plt.tight_layout()
plt.show()


In [ ]:
# ── Distribution of outcomes ─────────────────────────
all_prices = [p for row in matrix for p in row if not np.isnan(p)]

fig, ax = plt.subplots(figsize=(10, 4))

ax.hist(all_prices, bins=18, color=ACCENT, alpha=0.7, edgecolor='#0d1117', linewidth=0.8)
ax.axvline(intrinsic_price, color=GREEN,  linewidth=2, linestyle='-',  label=f'Base case: ${intrinsic_price:.2f}')
ax.axvline(current_price,   color=RED,    linewidth=2, linestyle='--', label=f'Market price: ${current_price:.2f}')
ax.axvline(np.median(all_prices), color=YELLOW, linewidth=1.5, linestyle=':', label=f'Median: ${np.median(all_prices):.2f}')

ax.set_title(f'{TICKER} — Distribution of Intrinsic Price Estimates across Sensitivity Grid',
             color='#e6edf3', fontsize=11)
ax.set_xlabel('Intrinsic Price ($)', color=MUTED)
ax.set_ylabel('Count of (WACC, TGR) combinations', color=MUTED)
ax.legend(fontsize=10)
ax.grid(axis='y')
plt.tight_layout()
plt.show()

print(f'Sensitivity range: ${min(all_prices):.2f} – ${max(all_prices):.2f}')
print(f'Median estimate  : ${np.median(all_prices):.2f}')
print(f'Base case        : ${intrinsic_price:.2f}')
print(f'Market price     : ${current_price:.2f}')


---
## 12. Results Summary & Interpretation

A complete results card and a plain-language interpretation of the output.


In [ ]:
print('=' * 60)
print(f'  DCF VALUATION SUMMARY — {TICKER}')
print('=' * 60)

company_name = _info.get('longName', TICKER)
print(f'  Company              : {company_name}')
print(f'  Ticker               : {TICKER}')
print()
print('  ── ASSUMPTIONS ──────────────────────────────────')
print(f'  Revenue Growth (CAGR): {REVENUE_GROWTH_RATE:.1%} / year')
print(f'  WACC                 : {WACC:.1%}')
print(f'  Terminal Growth Rate : {TERMINAL_GROWTH_RATE:.1%}')
print(f'  Tax Rate             : {TAX_RATE:.1%}')
print(f'  Forecast Horizon     : {FORECAST_YEARS} years')
print(f'  Trailing Avg Window  : {TRAILING_YEARS} years')
print()
print('  ── DERIVED MARGINS ──────────────────────────────')
print(f'  COGS Margin          : {cogs_margin:.1%}')
print(f'  OpEx Margin          : {opex_margin:.1%}')
print(f'  EBIT Margin          : {ebit_margin:.1%} (derived)')
print(f'  D&A Margin           : {da_margin:.1%}')
print(f'  CapEx Margin         : {capex_margin:.1%}')
print(f'  WC / Revenue         : {wc_margin:.1%}')
print()
print('  ── DCF OUTPUT ───────────────────────────────────')
print(f'  Σ PV of FCFs         : ${sum_pv_fcf:.2f}B')
print(f'  Terminal Value (TV)  : ${terminal_value:.2f}B  (undiscounted)')
print(f'  PV(TV)               : ${pv_tv:.2f}B')
print(f'  TV as % of EV        : {tv_pct_of_ev:.1%}')
print(f'  Enterprise Value     : ${enterprise_value:.2f}B')
print(f'  Net Debt             : ${net_debt:.2f}B')
print(f'  Equity Value         : ${equity_value:.2f}B')
print(f'  Shares (diluted)     : {shares_bn:.2f}B')
print()
print('  ── VERDICT ──────────────────────────────────────')
print(f'  Intrinsic Price      : ${intrinsic_price:.2f}')
print(f'  Current Market Price : ${current_price:.2f}')
sign_str = '+' if upside >= 0 else ''
print(f'  Upside / Downside    : {sign_str}{upside:.1%}')
verdict = ('UNDERVALUED ▲' if upside > 0.10
           else 'OVERVALUED ▼' if upside < -0.10
           else 'APPROXIMATELY FAIRLY VALUED ◆')
print(f'  Verdict              : {verdict}')
print()
sens_prices = [p for row in matrix for p in row if not np.isnan(p)]
print('  ── SENSITIVITY RANGE ────────────────────────────')
print(f'  Across all (WACC, TGR) combos:')
print(f'  Min estimate         : ${min(sens_prices):.2f}')
print(f'  Median estimate      : ${np.median(sens_prices):.2f}')
print(f'  Max estimate         : ${max(sens_prices):.2f}')
print('=' * 60)


### How to read this output

**The intrinsic price is not a precise prediction.** It is the present value
of this company's cash flows *under your specific assumptions*.
The sensitivity range is more informative than the single base-case number.

Key questions to ask yourself about the result:

1. **Is my revenue growth assumption realistic?**
   Compare it to the historical CAGR (Section 3) and to management guidance.
   A 3% variance in growth rate can shift intrinsic price by 15–25%.

2. **Are the margins stable?**
   If the Historical tab shows wide margin swings, the trailing average may
   not be a reliable anchor. Consider adjusting margins manually.

3. **Does TV dominate?**
   If TV > 75% of EV, the result is very sensitive to WACC and terminal
   growth rate. Extend the forecast horizon or widen the sensitivity range.

4. **What's the margin of safety?**
   Value investors typically require a 20–30% discount to intrinsic value
   before buying. A stock trading at 5% below intrinsic value is not a
   compelling case given model uncertainty.

5. **What does the sensitivity matrix say at your most pessimistic assumptions?**
   If even the most conservative (WACC, TGR) cell still shows upside,
   the bull case is robust.

---

> **Disclaimer:** This model is for educational and research purposes.
> It does not constitute investment advice. Financial data sourced from
> Yahoo Finance via the `yfinance` library. Always verify with official filings.
